In [9]:
# =========================================
# BLOCK 1 : IMPORTS + DATASET LOADING
# =========================================
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
import numpy as np
# Load Diabetes Dataset
X, y = load_diabetes(return_X_y=True)
print("Shape of X:", X.shape)
print("Shape of y:", y.shape)
# Train Test Split
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=2)
print("\nTraining Data Shape:", X_train.shape)
print("Testing Data Shape:", X_test.shape)

Shape of X: (442, 10)
Shape of y: (442,)

Training Data Shape: (353, 10)
Testing Data Shape: (89, 10)


In [13]:
# =========================================
# BLOCK 2 : UNIVERSAL GD CLASS
# Supports:
# 1. Batch Gradient Descent
# 2. Stochastic Gradient Descent
# 3. Mini Batch Gradient Descent
# =========================================

class UniversalGD:
    def __init__(self,mode='batch',batch_size=None,learning_rate=0.01,epochs=100):
        self.mode = mode
        self.batch_size = batch_size
        self.lr = learning_rate
        self.epochs = epochs
        self.coef_ = None
        self.intercept_ = None
    def fit(self, X_train, y_train):
        # Step 1: Initialize parameters
        self.intercept_ = 0
        self.coef_ = np.ones(X_train.shape[1])
        # Step 2: Training Loop
        for i in range(self.epochs):
            # =================================
            # BATCH GRADIENT DESCENT
            # =================================
            if self.mode == 'batch':
                y_hat = np.dot(X_train, self.coef_) + self.intercept_
                intercept_der = -2 * np.mean(y_train - y_hat)
                coef_der = -2 * np.dot((y_train - y_hat),X_train) / X_train.shape[0]
                self.intercept_ = self.intercept_ - self.lr * intercept_der
                self.coef_ = self.coef_ - self.lr * coef_der
            # =================================
            # STOCHASTIC GRADIENT DESCENT
            # =================================
            elif self.mode == 'sgd':
                for j in range(X_train.shape[0]):
                    idx = np.random.randint(0,X_train.shape[0])
                    x_i = X_train[idx]
                    y_i = y_train[idx]
                    y_hat = np.dot(x_i,self.coef_) + self.intercept_
                    intercept_der = -2 * (y_i - y_hat)
                    coef_der = -2 * np.dot((y_i - y_hat),x_i)
                    self.intercept_ = self.intercept_ - self.lr * intercept_der
                    self.coef_ = self.coef_ - self.lr * coef_der
            # =================================
            # MINI BATCH GRADIENT DESCENT
            # =================================
            elif self.mode == 'mini_batch':
                total_batches = int(X_train.shape[0] / self.batch_size)
                for j in range(total_batches):
                    idx = np.random.choice(X_train.shape[0],self.batch_size,replace=False)
                    x_batch = X_train[idx]
                    y_batch = y_train[idx]
                    y_hat = np.dot(x_batch,self.coef_) + self.intercept_
                    intercept_der = -2 * np.mean(y_batch - y_hat)
                    coef_der = -2 * np.dot((y_batch - y_hat),x_batch) / self.batch_size
                    self.intercept_ = self.intercept_ - self.lr * intercept_der
                    self.coef_ = self.coef_ - self.lr * coef_der
        print("\nTraining Completed")
        print("Intercept:", self.intercept_)
        print("Coefficients:", self.coef_)
    def predict(self, X_test):
        return np.dot(X_test,self.coef_) + self.intercept_

In [4]:
# =========================================
# BLOCK 3 : RUN BATCH GRADIENT DESCENT
# =========================================

bgd = UniversalGD(
    mode='batch',
    learning_rate=0.5,
    epochs=1000
)

bgd.fit(X_train, y_train)

y_pred_batch = bgd.predict(X_test)

print("\nBatch GD R2 Score:")
print(r2_score(y_test, y_pred_batch))


Training Completed
Intercept: 152.01351687661833
Coefficients: [  14.38990585 -173.7235727   491.54898524  323.91524824  -39.32648042
 -116.01061213 -194.04077415  103.38135565  451.63448787   97.57218278]

Batch GD R2 Score:
0.4534503034722803


In [5]:
# =========================================
# BLOCK 4 : RUN STOCHASTIC GD
# =========================================

sgd = UniversalGD(
    mode='sgd',
    learning_rate=0.01,
    epochs=40
)

sgd.fit(X_train, y_train)

y_pred_sgd = sgd.predict(X_test)

print("\nSGD R2 Score:")
print(r2_score(y_test, y_pred_sgd))


Training Completed
Intercept: 149.12773926762438
Coefficients: [  68.49440389  -49.88131418  310.01840807  225.67541335   35.62568878
   -4.77231606 -156.90799293  129.68745963  291.62757176  131.1085885 ]

SGD R2 Score:
0.41661936246061104


In [6]:
# =========================================
# BLOCK 5 : RUN MINI BATCH GD
# =========================================

mbgd = UniversalGD(
    mode='mini_batch',
    batch_size=32,   # Common batch size
    learning_rate=0.01,
    epochs=100
)

mbgd.fit(X_train, y_train)

y_pred_mbgd = mbgd.predict(X_test)

print("\nMini Batch GD R2 Score:")
print(r2_score(y_test, y_pred_mbgd))


Training Completed
Intercept: 151.75904088270354
Coefficients: [ 16.97877693   3.23823776  45.46788635  35.2761659   16.49400822
  12.72554245 -27.29792889  30.54173648  45.66855886  26.86525815]

Mini Batch GD R2 Score:
0.1159034771897609


In [15]:
# =========================================
# BLOCK 6 : COMPARISON WITH SCIKIT-LEARN
# =========================================

from sklearn.linear_model import LinearRegression
from sklearn.linear_model import SGDRegressor


# Batch-like Exact Solution
reg = LinearRegression()
reg.fit(X_train, y_train)

y_pred = reg.predict(X_test)

print("\nScikit-Learn LinearRegression R2 Score:")
print(r2_score(y_test, y_pred))


# SGD using Scikit-Learn
sgd_reg = SGDRegressor(
    max_iter=1000,
    learning_rate='constant',
    eta0=0.01
)

sgd_reg.fit(X_train, y_train)

y_pred = sgd_reg.predict(X_test)

print("\nScikit-Learn SGDRegressor R2 Scor e:")
print(r2_score(y_test, y_pred))


Scikit-Learn LinearRegression R2 Score:
0.439933866156897

Scikit-Learn SGDRegressor R2 Scor e:
0.43024866468543643
